# 📐 Modelado de Datos — Persistencia en Hive
**Materia:** Herramientas de Software para Big Data  

**Modelo:** Normalizado  
**Fuente:** `/rfn/obligatorio/`  
**Destino:** Tablas Hive en esquema `inumet`

**Tablas del modelo:**
```
inumet.dim_estaciones     — datos de cada estacion meteorologica
inumet.dim_tiempo         — dimension temporal con atributos derivados
inumet.fact_temperatura   — mediciones de temperatura del aire
inumet.fact_viento        — mediciones de intensidad y direccion del viento
inumet.fact_precipitacion — mediciones de precipitacion horaria
inumet.fact_humedad       — mediciones de humedad relativa
inumet.fact_presion       — mediciones de presion atmosferica
inumet.fact_insolacion    — mediciones de horas de insolacion solar
```

## 1. Inicializacion de Spark con soporte Hive

In [ ]:
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder \
    .appName("INUMET_Modelado_Hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo con soporte Hive")

## 2. Creacion del esquema en Hive

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS inumet")
spark.sql("USE inumet")

print("Esquemas disponibles en Hive:")
spark.sql("SHOW DATABASES").show()

## 3. Carga desde /rfn

In [ ]:
RFN = "hdfs://localhost:9000/rfn/obligatorio"

fuentes = {
    "temperatura": "temperatura",
    "viento": "viento",
    "precipitacion": "precipitacion",
    "humedad": "humedad",
    "presion": "presion",
    "heliofania": "heliofania",
}

tablas_rfn = {
    nombre: spark.read.parquet(f"{RFN}/{carpeta}").cache()
    for nombre, carpeta in fuentes.items()
}

df_temp    = tablas_rfn["temperatura"]
df_viento  = tablas_rfn["viento"]
df_lluvia  = tablas_rfn["precipitacion"]
df_humedad = tablas_rfn["humedad"]
df_presion = tablas_rfn["presion"]
df_helio   = tablas_rfn["heliofania"]

In [ ]:
print("Tablas cargadas desde /rfn:")
for nombre, df in tablas_rfn.items():
    print(f"  {nombre:<15}: {df.count():>10,} registros")

## 4. dim_estaciones
Construida manualmente con los datos conocidos de cada estacion.

In [ ]:
estaciones_data = [
    ("Aeropuerto Melilla G3", "Montevideo",  "costera",  -34.8335, -56.0303),
    ("Artigas G3",            "Artigas",     "interior", -30.4000, -56.5000),
    ("Colonia G3",            "Colonia",     "costera",  -34.4622, -57.8408),
    ("Mercedes G3",           "Soriano",     "interior", -33.2500, -58.0833),
    ("Paso de los Toros G3",  "Tacuarembo",  "interior", -32.8167, -56.5167),
    ("Rocha G3",              "Rocha",       "costera",  -34.4833, -54.3333),
    ("Salto G3",              "Salto",       "interior", -31.3833, -57.9667),
]

schema_est = StructType([
    StructField("estacion_id",  StringType(), False),
    StructField("departamento", StringType(), False),
    StructField("zona",         StringType(), False),
    StructField("latitud",      DoubleType(), True),
    StructField("longitud",     DoubleType(), True),
])

dim_estaciones = spark.createDataFrame(estaciones_data, schema=schema_est).cache()
dim_estaciones.show(truncate=False)

## 5. dim_tiempo
Construida a partir de todas las fechas unicas de los datos.

In [ ]:
fechas = reduce(
    lambda a, b: a.unionByName(b),
    [df.select("fecha") for df in tablas_rfn.values()]
).dropDuplicates(["fecha"])

meses_expr = F.create_map(*[
    x for par in [
        (F.lit(1), F.lit("Enero")),
        (F.lit(2), F.lit("Febrero")),
        (F.lit(3), F.lit("Marzo")),
        (F.lit(4), F.lit("Abril")),
        (F.lit(5), F.lit("Mayo")),
        (F.lit(6), F.lit("Junio")),
        (F.lit(7), F.lit("Julio")),
        (F.lit(8), F.lit("Agosto")),
        (F.lit(9), F.lit("Septiembre")),
        (F.lit(10), F.lit("Octubre")),
        (F.lit(11), F.lit("Noviembre")),
        (F.lit(12), F.lit("Diciembre")),
    ] for x in par
])

dim_tiempo = fechas \
    .withColumn("anio", F.year("fecha")) \
    .withColumn("mes",  F.month("fecha")) \
    .withColumn("dia",  F.dayofmonth("fecha")) \
    .withColumn("hora", F.hour("fecha")) \
    .withColumn(
        "estacion_anio",
        F.when((F.col("mes") >= 12) | (F.col("mes") <= 2), "verano")
         .when((F.col("mes") >= 3) & (F.col("mes") <= 5), "otonio")
         .when((F.col("mes") >= 6) & (F.col("mes") <= 8), "invierno")
         .otherwise("primavera")
    ) \
    .withColumn("nombre_mes", meses_expr[F.col("mes")]) \
    .orderBy("fecha") \
    .cache()

In [ ]:
print(f"dim_tiempo: {dim_tiempo.count():,} registros unicos")
dim_tiempo.show(5, truncate=False)

## 6. Tablas de hechos

In [ ]:
config_hechos = {
    "fact_temperatura": {
        "fuente": "temperatura",
        "columnas": ["fecha", "estacion_id", "temp_aire"],
    },
    "fact_viento": {
        "fuente": "viento",
        "columnas": ["fecha", "estacion_id", "int_viento", "dir_viento"],
    },
    "fact_precipitacion": {
        "fuente": "precipitacion",
        "columnas": ["fecha", "estacion_id", "precip_horario"],
    },
    "fact_humedad": {
        "fuente": "humedad",
        "columnas": ["fecha", "estacion_id", "hum_relativa"],
    },
    "fact_presion": {
        "fuente": "presion",
        "columnas": ["fecha", "estacion_id", "pres_atm_mar"],
    },
    "fact_insolacion": {
        "fuente": "heliofania",
        "columnas": ["fecha", "estacion_id"],
        "renombres": {"heliofania": "horas_insolacion"},
    },
}

def construir_hecho(cfg):
    df = tablas_rfn[cfg["fuente"]]
    columnas = [F.col(c) for c in cfg["columnas"]]
    for origen, destino in cfg.get("renombres", {}).items():
        columnas.append(F.col(origen).alias(destino))
    return df.select(*columnas)

hechos = {
    nombre: construir_hecho(cfg).cache()
    for nombre, cfg in config_hechos.items()
}

fact_temperatura   = hechos["fact_temperatura"]
fact_viento        = hechos["fact_viento"]
fact_precipitacion = hechos["fact_precipitacion"]
fact_humedad       = hechos["fact_humedad"]
fact_presion       = hechos["fact_presion"]
fact_insolacion    = hechos["fact_insolacion"]

In [ ]:
print("Tablas de hechos creadas:")
for nombre, df in hechos.items():
    print(f"  {nombre:<25}: {df.count():>10,} registros | {df.columns}")

## 7. Persistencia en Hive
Se guardan todas las tablas en el esquema `inumet` de Hive.

In [ ]:
tablas_hive = {
    "inumet.dim_estaciones":     dim_estaciones,
    "inumet.dim_tiempo":         dim_tiempo,
    "inumet.fact_temperatura":   fact_temperatura,
    "inumet.fact_viento":        fact_viento,
    "inumet.fact_precipitacion": fact_precipitacion,
    "inumet.fact_humedad":       fact_humedad,
    "inumet.fact_presion":       fact_presion,
    "inumet.fact_insolacion":    fact_insolacion,
}

for nombre_tabla, df in tablas_hive.items():
    df.write.mode("overwrite").format("parquet").saveAsTable(nombre_tabla)
    print(f"  ✅ {nombre_tabla}")

print("
Todas las tablas guardadas en Hive correctamente")

## 8. Verificacion — tablas en Hive

In [ ]:
print("Tablas en el esquema inumet:")
spark.sql("SHOW TABLES IN inumet").show(truncate=False)

In [ ]:
print("Conteo de registros por tabla:")
print(f"{'Tabla':<30} {'Registros':>12}")
print("-"*44)

for nombre_tabla in tablas_hive.keys():
    n = spark.table(nombre_tabla).count()
    print(f"  {nombre_tabla:<28} {n:>12,}")

## 9. Vista previa de las tablas en Hive
Consultamos las tablas directamente desde Hive para confirmar que funcionan.

In [ ]:
print("=== inumet.dim_estaciones ===")
spark.table("inumet.dim_estaciones").show(truncate=False)

print("
=== inumet.dim_tiempo (muestra) ===")
spark.table("inumet.dim_tiempo").show(5, truncate=False)

In [ ]:
print("=== inumet.fact_temperatura (muestra) ===")
spark.table("inumet.fact_temperatura").show(5, truncate=False)

print("
=== inumet.fact_insolacion (muestra) ===")
spark.table("inumet.fact_insolacion").show(5, truncate=False)

In [ ]:
spark.stop()
print("Sesion Spark cerrada.")